# Remote Execution with Ray - Interactive Demo

This notebook demonstrates how to use **Burr's Action Execution Interceptors** to run actions on Ray workers.

## What You'll Learn

1. How to create a Ray interceptor
2. How to define orchestrator vs. worker hooks
3. How to selectively run actions locally vs. remotely
4. How state flows between main process and Ray workers

## Setup

In [ ]:
import time
from typing import Dict, Any, Optional

import ray
from burr.core import Action, State, ApplicationBuilder, action
from burr.lifecycle import (
    ActionExecutionInterceptorHook,
    PreRunStepHookWorker,
    PostRunStepHookWorker,
    PreRunStepHook,
    PostRunStepHook,
)

## Step 1: Define Actions

We'll create three actions:
- `increment_local` - runs locally (no `ray` tag)
- `heavy_computation` - runs on Ray (tagged with `ray`)
- `another_ray_task` - also runs on Ray

In [ ]:
@action(reads=["count"], writes=["count", "last_operation"], tags=["local"])
def increment_local(state: State) -> tuple:
    """Increment counter locally (not on Ray)"""
    result = {
        "count": state["count"] + 1,
        "last_operation": "increment_local",
    }
    return result, state.update(**result)


@action(reads=["count"], writes=["count", "last_operation"], tags=["ray"])
def heavy_computation(state: State, multiplier: int = 2) -> tuple:
    """Simulate heavy computation that should run on Ray"""
    print(f"🔧 [Ray Worker] Running heavy computation with multiplier={multiplier}")
    time.sleep(0.5)  # Simulate work
    result = {
        "count": state["count"] * multiplier,
        "last_operation": f"heavy_computation(x{multiplier})",
    }
    return result, state.update(**result)


@action(reads=["count"], writes=["count", "last_operation"], tags=["ray"])
def another_ray_task(state: State) -> tuple:
    """Another task that runs on Ray"""
    print("🔧 [Ray Worker] Running another Ray task")
    time.sleep(0.3)  # Simulate work
    result = {
        "count": state["count"] + 10,
        "last_operation": "another_ray_task(+10)",
    }
    return result, state.update(**result)

## Step 2: Define Hooks

We define two types of hooks:
1. **Orchestrator hooks** - run on the main process
2. **Worker hooks** - run on Ray workers

In [ ]:
# Orchestrator hooks (run on main process)
class OrchestratorPreHook(PreRunStepHook):
    def pre_run_step(self, *, action: Action, state: State, inputs: Dict[str, Any], **kwargs):
        print(f"📋 [Main Process] About to execute: {action.name}")


class OrchestratorPostHook(PostRunStepHook):
    def post_run_step(
        self, *, action: Action, state: State, result: Optional[Dict[str, Any]], exception, **kwargs
    ):
        print(f"✅ [Main Process] Finished: {action.name}")


# Worker hooks (run on Ray workers)
class WorkerPreHook(PreRunStepHookWorker):
    def pre_run_step_worker(self, *, action: Action, state: State, inputs: Dict[str, Any], **kwargs):
        print(f"⚙️  [Ray Worker] Starting: {action.name}")


class WorkerPostHook(PostRunStepHookWorker):
    def post_run_step_worker(
        self, *, action: Action, state: State, result: Optional[Dict[str, Any]], exception, **kwargs
    ):
        print(f"✨ [Ray Worker] Completed: {action.name}")

## Step 3: Create the Ray Interceptor

The interceptor decides which actions to run on Ray and handles the remote execution.

In [ ]:
class RayActionInterceptor(ActionExecutionInterceptorHook):
    """Interceptor that executes actions tagged with 'ray' on Ray workers"""

    def __init__(self):
        self.ray_initialized = False

    def _ensure_ray_initialized(self):
        if not self.ray_initialized:
            if not ray.is_initialized():
                print("🚀 [Main Process] Initializing Ray...")
                ray.init(ignore_reinit_error=True)
            self.ray_initialized = True

    def should_intercept(self, *, action: Action, **kwargs) -> bool:
        """Intercept actions tagged with 'ray'"""
        return "ray" in action.tags

    def intercept_run(
        self, *, action: Action, state: State, inputs: Dict[str, Any], **kwargs
    ) -> dict:
        """Execute the action on a Ray worker"""
        self._ensure_ray_initialized()

        print(f"📤 [Main Process] Dispatching {action.name} to Ray...")

        # Extract worker hooks
        worker_adapter_set = kwargs.get("worker_adapter_set")

        # Create a Ray remote function
        @ray.remote
        def execute_on_ray():
            # Call pre-worker hooks
            if worker_adapter_set:
                worker_adapter_set.call_all_lifecycle_hooks_sync(
                    "pre_run_step_worker",
                    action=action,
                    state=state,
                    inputs=inputs,
                )

            # Execute the action
            if hasattr(action, "single_step") and action.single_step:
                result, new_state = action.run_and_update(state, **inputs)
            else:
                state_to_use = state.subset(*action.reads)
                result = action.run(state_to_use, **inputs)
                new_state = None

            # Call post-worker hooks
            if worker_adapter_set:
                worker_adapter_set.call_all_lifecycle_hooks_sync(
                    "post_run_step_worker",
                    action=action,
                    state=state,
                    result=result,
                    exception=None,
                )

            return result, new_state

        # Execute remotely and wait for result
        result_ref = execute_on_ray.remote()
        result, new_state = ray.get(result_ref)

        print(f"📥 [Main Process] Received result from Ray for {action.name}")

        # For single-step actions, include the new state
        if new_state is not None:
            result_with_state = result.copy()
            result_with_state["__INTERCEPTOR_NEW_STATE__"] = new_state
            return result_with_state

        return result

## Step 4: Build the Application

Now we put it all together!

In [ ]:
# Create interceptor and hooks
ray_interceptor = RayActionInterceptor()
orchestrator_pre = OrchestratorPreHook()
orchestrator_post = OrchestratorPostHook()
worker_pre = WorkerPreHook()
worker_post = WorkerPostHook()

# Build the application
app = (
    ApplicationBuilder()
    .with_state(count=0)
    .with_actions(
        increment_local,
        heavy_computation,
        another_ray_task,
    )
    .with_transitions(
        ("increment_local", "heavy_computation"),
        ("heavy_computation", "another_ray_task"),
        ("another_ray_task", "increment_local"),
    )
    .with_entrypoint("increment_local")
    .with_hooks(
        ray_interceptor,
        orchestrator_pre,
        orchestrator_post,
        worker_pre,
        worker_post,
    )
    .build()
)

print("✨ Application built successfully!")

## Step 5: Execute Actions

Let's run through the workflow step by step.

### Step 5a: Local Execution

In [ ]:
print("\n" + "="*60)
print("STEP 1: Local Execution (increment_local)")
print("="*60)

action, result, state = app.step()

print(f"\n📊 Result: count={state['count']}, operation={state['last_operation']}")

Notice:
- ✅ Orchestrator hooks run
- ❌ Worker hooks DON'T run (action not intercepted)
- ❌ No Ray dispatch messages

### Step 5b: Ray Execution #1

In [ ]:
print("\n" + "="*60)
print("STEP 2: Ray Execution (heavy_computation)")
print("="*60)

action, result, state = app.step(inputs={"multiplier": 3})

print(f"\n📊 Result: count={state['count']}, operation={state['last_operation']}")

Notice:
- ✅ Orchestrator hooks run (on main process)
- ✅ Worker hooks run (on Ray worker!)
- ✅ Ray dispatch and receive messages
- ✅ Actual computation happens on Ray worker

### Step 5c: Ray Execution #2

In [ ]:
print("\n" + "="*60)
print("STEP 3: Ray Execution (another_ray_task)")
print("="*60)

action, result, state = app.step()

print(f"\n📊 Result: count={state['count']}, operation={state['last_operation']}")

### Step 5d: Back to Local

In [ ]:
print("\n" + "="*60)
print("STEP 4: Back to Local Execution (increment_local)")
print("="*60)

action, result, state = app.step()

print(f"\n📊 Result: count={state['count']}, operation={state['last_operation']}")

## Step 6: View Final State

In [ ]:
print("\n" + "="*60)
print("FINAL STATE")
print("="*60)
print(f"Count: {state['count']}")
print(f"Last Operation: {state['last_operation']}")
print("\nWorkflow: 0 → +1 (local) → x3 (ray) → +10 (ray) → +1 (local) = 4")

## Cleanup

In [ ]:
if ray.is_initialized():
    print("🛑 Shutting down Ray...")
    ray.shutdown()
    print("✅ Ray shutdown complete")

## Key Takeaways

1. **Selective Execution**: Actions can run locally or remotely based on tags
2. **Two-Tier Hooks**: Orchestrator hooks always run; worker hooks only run for intercepted actions
3. **Seamless Integration**: State flows naturally between main process and workers
4. **Transparent to Actions**: Actions don't know they're running on Ray
5. **Flexible**: Easy to add more actions or change execution backend

## Next Steps

Try modifying this notebook:
- Add your own actions with different tags
- Create a more complex workflow
- Add custom logging in the hooks
- Experiment with async actions
- Try with actual compute-intensive operations